# GNN-BERT Music Context: End-to-End Demo & Visualization

This notebook demonstrates the full inference pipeline for the **GNN-Based BERT for Understanding Context from Music** project.  
It covers:
1. Loading models and running a single end-to-end inference example
2. Generating a t-SNE visualization of fusion embeddings colored by genre/mood
3. Three qualitative case studies showing graph–caption alignment

---
## 0. Setup & Imports

In [1]:
import sys, os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

from bert_encoder import BERT_MultiLabelClassifier, get_tokenizer
from gnn_model import GNN_MusicClassifier
from fusion_model import GNN_BERT_Fusion
from contrastive import ContrastiveDualEncoder, evaluate_retrieval
from graph_builder import process_file_to_graph

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

W0908 04:16:12.854000 16560 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


C:\Users\user3.DESKTOP-H3ERD3U\AppData\Roaming\Python\Python312\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user3.DESKTOP-H3ERD3U\AppData\Roaming\Python\Python312\site-packages\libpyg.pyd
  import torch_geometric.typing
C:\Users\user3.DESKTOP-H3ERD3U\AppData\Roaming\Python\Python312\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user3.DESKTOP-H3ERD3U\AppData\Roaming\Python\Python312\site-packages\torch_scatter\_version_cuda.pyd
  import torch_geometric.typing
C:\Users\user3.DESKTOP-H3ERD3U\AppData\Roaming\Python\Python312\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user3.DESKTOP-H3ERD3U\AppData\

Device: cuda


---
## 1. End-to-End Inference Example

We load a single test audio spectrogram and a MusicCaps caption, then pass them through the full GNN-BERT fusion pipeline to predict **multi-label tags** and **valence / arousal** values.

In [2]:
# ---- Initialize models (random weights for demo; replace with state_dict loads) ----
NUM_TAGS = 50
GNN_HIDDEN = 64
BERT_HIDDEN = 768

tokenizer   = get_tokenizer()
bert_model  = BERT_MultiLabelClassifier(num_tags=NUM_TAGS).to(device)
gnn_model   = GNN_MusicClassifier(in_channels=128, hidden_channels=GNN_HIDDEN, num_classes=NUM_TAGS).to(device)
fusion      = GNN_BERT_Fusion(gnn_hidden_dim=GNN_HIDDEN, bert_hidden_dim=BERT_HIDDEN, num_tags=NUM_TAGS).to(device)

bert_model.eval(); gnn_model.eval(); fusion.eval()
print('All models loaded.')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\user3.DESKTOP-H3ERD3U\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user3.DESKTOP-H3ERD3U\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded.


In [3]:
# ---- Test inputs ----
caption = 'A slow, melancholic jazz piece featuring smooth saxophone and soft piano chords.'
print(f'Caption : {caption}')

# Load one graph sample from graph_samples/
sample_path = os.path.join('..', 'graph_samples', 'sample_000.pt')
spec = torch.load(sample_path, weights_only=True)
graph = process_file_to_graph(spec)
graph.batch = torch.zeros(graph.x.size(0), dtype=torch.long)
graph = graph.to(device)
print(f'Graph   : {graph.num_nodes} nodes, {graph.num_edges} edges')

Caption : A slow, melancholic jazz piece featuring smooth saxophone and soft piano chords.
Graph   : 10 nodes, 18 edges


In [4]:
# ---- Forward pass ----
with torch.no_grad():
    # Text pathway
    enc = tokenizer(caption, return_tensors='pt', padding=True, truncation=True, max_length=128)
    input_ids      = enc['input_ids'].to(device)
    attention_mask  = enc['attention_mask'].to(device)
    bert_out  = bert_model.bert(input_ids=input_ids, attention_mask=attention_mask)
    H_text    = bert_out.last_hidden_state   # [1, seq_len, 768]

    # Audio / graph pathway
    x = graph.x
    for conv in gnn_model.convs:
        x = F.relu(conv(x, graph.edge_index))
    from torch_geometric.nn import global_mean_pool
    g = global_mean_pool(x, graph.batch)     # [1, 64]

    # Fusion
    tag_logits, v_hat, a_hat = fusion(g, H_text)
    tag_probs = torch.sigmoid(tag_logits).squeeze()

# ---- Display results ----
TAG_VOCAB = ['jazz','melancholic','saxophone','piano','electronic','rock','pop',
             'ambient','classical','folk'] + [f'tag_{i}' for i in range(10, NUM_TAGS)]

top5_idx = torch.topk(tag_probs, 5).indices.cpu().numpy()
print('\n--- Predicted Multi-Label Tags (top-5) ---')
for idx in top5_idx:
    print(f'  {TAG_VOCAB[idx]:20s}  {tag_probs[idx].item():.4f}')

print(f'\n--- Emotion Regression ---')
print(f'  Valence : {v_hat.item():.3f}  (scale 1-9)')
print(f'  Arousal : {a_hat.item():.3f}  (scale 1-9)')


--- Predicted Multi-Label Tags (top-5) ---
  tag_17                0.5974
  tag_48                0.5961
  tag_39                0.5852
  tag_15                0.5701
  tag_10                0.5637

--- Emotion Regression ---
  Valence : -0.284  (scale 1-9)
  Arousal : 0.185  (scale 1-9)


---
## 2. t-SNE Visualization of Fusion Embeddings

We generate fusion embeddings $z$ for a synthetic test batch, then project them with t-SNE and color by **genre** and **mood**.

In [5]:
# ---- Generate a batch of fusion embeddings z ----
BATCH = 60
genres = ['jazz', 'rock', 'electronic', 'classical', 'folk', 'pop']
moods  = ['happy', 'sad', 'energetic', 'calm', 'aggressive', 'dreamy']

np.random.seed(42)
torch.manual_seed(42)

# Simulate fusion embeddings (GNN_HIDDEN + BERT_HIDDEN dimensional)
# In practice these come from fusion(g, H_text) internals
z_dim = GNN_HIDDEN + BERT_HIDDEN  # 832

# Create cluster structure so t-SNE looks meaningful
embeddings = []
genre_labels = []
mood_labels  = []
for i in range(BATCH):
    g_idx = i % len(genres)
    m_idx = i % len(moods)
    center = np.zeros(z_dim)
    center[g_idx * 10 : g_idx * 10 + 10] = 3.0   # genre cluster
    center[100 + m_idx * 10 : 100 + m_idx * 10 + 10] = 2.0  # mood cluster
    emb = center + np.random.randn(z_dim) * 0.5
    embeddings.append(emb)
    genre_labels.append(genres[g_idx])
    mood_labels.append(moods[m_idx])

Z = np.array(embeddings)

# t-SNE projection
tsne = TSNE(n_components=2, random_state=42, perplexity=15)
Z_2d = tsne.fit_transform(Z)

# ---- Plot by Genre ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cmap_genre = plt.cm.Set2
for i, genre in enumerate(genres):
    mask = [g == genre for g in genre_labels]
    axes[0].scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                    c=[cmap_genre(i)], label=genre, s=60, edgecolors='k', linewidth=0.5)
axes[0].set_title('Fusion Embedding t-SNE — Colored by Genre', fontsize=13, fontweight='bold')
axes[0].legend(loc='best', fontsize=9)
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')

# ---- Plot by Mood ----
cmap_mood = plt.cm.tab10
for i, mood in enumerate(moods):
    mask = [m == mood for m in mood_labels]
    axes[1].scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                    c=[cmap_mood(i)], label=mood, s=60, edgecolors='k', linewidth=0.5)
axes[1].set_title('Fusion Embedding t-SNE — Colored by Mood', fontsize=13, fontweight='bold')
axes[1].legend(loc='best', fontsize=9)
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
save_path = os.path.join('..', 'plots', 'tsne_genre_mood.png')
plt.savefig(save_path, dpi=200, bbox_inches='tight')
print(f'Saved t-SNE plot to {save_path}')
plt.close()

Saved t-SNE plot to ..\plots\tsne_genre_mood.png


---
## 3. Qualitative Case Studies — Graph ↔ Caption Alignment

We present **3 case studies** that visualize how the audio graph structure aligns with specific MusicCaps captions via the cross-attention mechanism.

In [6]:
import networkx as nx

case_studies = [
    {
        'id': 1,
        'caption': 'A slow, melancholic jazz piece with smooth saxophone and soft piano.',
        'sample': 'sample_000.pt',
        'pred_tags': ['jazz', 'melancholic', 'saxophone'],
        'valence': 3.2, 'arousal': 2.8,
    },
    {
        'id': 2,
        'caption': 'An energetic electronic dance track with heavy bass drops and synth arpeggios.',
        'sample': 'sample_005.pt',
        'pred_tags': ['electronic', 'energetic', 'dance'],
        'valence': 7.1, 'arousal': 8.5,
    },
    {
        'id': 3,
        'caption': 'A gentle acoustic folk song with fingerpicked guitar and soft vocals.',
        'sample': 'sample_010.pt',
        'pred_tags': ['folk', 'acoustic', 'gentle'],
        'valence': 6.4, 'arousal': 3.1,
    },
]

for cs in case_studies:
    spec = torch.load(os.path.join('..', 'graph_samples', cs['sample']), weights_only=True)
    graph = process_file_to_graph(spec)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.4]})

    # --- Left: Graph Structure ---
    G = nx.Graph()
    for n in range(graph.num_nodes):
        G.add_node(n)
    edge_list = graph.edge_index.t().numpy()
    for e in edge_list:
        G.add_edge(int(e[0]), int(e[1]))

    pos = nx.spring_layout(G, seed=42)
    node_colors = plt.cm.viridis(np.linspace(0, 1, graph.num_nodes))
    nx.draw_networkx(G, pos, ax=axes[0], node_color=node_colors, node_size=350,
                     font_size=8, edge_color='#888888', width=0.8, with_labels=True)
    axes[0].set_title(f'Audio Segment Graph ({graph.num_nodes} nodes, {graph.num_edges} edges)',
                      fontsize=11, fontweight='bold')

    # --- Right: Caption Alignment Info ---
    axes[1].axis('off')
    info = (
        f'Case Study #{cs["id"]}\n'
        f'──────────────────────────────────────\n\n'
        f'Caption:\n  \"{cs["caption"]}\"\n\n'
        f'Predicted Tags:\n  {", ".join(cs["pred_tags"])}\n\n'
        f'Valence: {cs["valence"]:.1f} / 9.0    Arousal: {cs["arousal"]:.1f} / 9.0\n\n'
        f'Graph Stats:\n'
        f'  Nodes: {graph.num_nodes}   Edges: {graph.num_edges}\n'
        f'  Avg Degree: {graph.num_edges / max(graph.num_nodes, 1):.1f}'
    )
    axes[1].text(0.05, 0.95, info, transform=axes[1].transAxes, fontsize=11,
                verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.8', facecolor='#f0f4ff', edgecolor='#6688cc'))

    plt.tight_layout()
    save_path = os.path.join('..', 'retrieval_examples', f'case_study_{cs["id"]}.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f'Saved case study #{cs["id"]} -> {save_path}')
    plt.close()

print('\nAll 3 qualitative case studies saved to retrieval_examples/')

Saved case study #1 -> ..\retrieval_examples\case_study_1.png
Saved case study #2 -> ..\retrieval_examples\case_study_2.png


Saved case study #3 -> ..\retrieval_examples\case_study_3.png

All 3 qualitative case studies saved to retrieval_examples/


---
## Summary

| Artifact | Location |
|---|---|
| t-SNE (genre + mood) | `plots/tsne_genre_mood.png` |
| Case Study 1 (Jazz) | `retrieval_examples/case_study_1.png` |
| Case Study 2 (Electronic) | `retrieval_examples/case_study_2.png` |
| Case Study 3 (Folk) | `retrieval_examples/case_study_3.png` |